# Load Mistral 7B (4-bit) and extract KPIs

This notebook installs required libraries, logs into Hugging Face, loads the Mistral-7B-Instruct model in 4-bit quantized mode, runs a prompt on a sample business case, and prints the extracted Business Goals, Key Stakeholders, and KPIs. An ELI5 explanation is included at the end.

In [ ]:
# !pip install torch torchvision torchaudio transformers accelerate bitsandbytes --quiet
# !pip install torch torchvision torchaudio transformers accelerate bitsandbytes --quiet

In [3]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(override=True)

# Get API key
api_key = os.getenv('HUGGINGFACE_TOKEN')
if not api_key:
    raise ValueError("HUGGINGFACE_TOKEN not found in .env file")

# Replace "your_token_here" with your Hugging Face access token
os.environ["HUGGINGFACE_TOKEN"] = api_key 
from huggingface_hub import login
login(token=os.environ["HUGGINGFACE_TOKEN"]) 

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.1"

quant_config = BitsAndBytesConfig(load_in_4bit=True)
tokenizer = AutoTokenizer.from_pretrained(model_name, token=os.environ["HUGGINGFACE_TOKEN"])
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant_config, token=os.environ["HUGGINGFACE_TOKEN"]) 

# Move model to GPU
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

RuntimeError: unable to mmap 9942981696 bytes from file </home/ridhwanlaptop/.cache/huggingface/hub/models--mistralai--Mistral-7B-Instruct-v0.1/snapshots/ec5deb64f2c6e6fa90c1abf74a91d5c93a9669ca/model-00001-of-00002.safetensors>: Cannot allocate memory (12)

In [ ]:
business_case = """
Our retail company is expanding rapidly and we need better visibility into sales performance across different regions.
We are currently using separate systems in each region and struggling to generate consolidated reports.
Our main goal is to improve decision-making by integrating all sales data into one centralized warehouse.
We also want to track KPIs like regional sales growth, top-performing products, and customer retention rates.
This project is sponsored by the Sales Director and supported by the Regional Managers.
"""

In [ ]:
#  STEP 5: Craft the prompt
prompt = f"""
You are a business analyst assistant. Given the following business case, extract:
1. Business Goals
2. Key Stakeholders
3. Important KPIs

Business Case:
{business_case}
"""

In [ ]:
#  STEP 6: Generate LLM response
inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
try:
    inputs = inputs.to(device)
except Exception:
    inputs = {k: v.to(device) for k, v in inputs.items()}
outputs = model.generate(**inputs, max_new_tokens=300)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

#  STEP 7: Print response
print(" LLM Extracted Output:\n")
print(response)

**ELI5 (Explain Like I'm 5):**

- **What this notebook does:** It installs model libraries, logs into Hugging Face, loads a large language model (Mistral 7B) in a compressed 4-bit format so it fits on smaller GPUs, feeds it a short business case, and asks the model to extract the business goals, stakeholders, and KPIs.
- **Why 4-bit?** It reduces GPU memory use so big models can run on less powerful hardware.
- **What you get at the end:** The model's plain-text answer with the extracted goals, stakeholders, and KPIs printed to the notebook output.
- **Important note:** Replace `your_token_here` with your Hugging Face token before running.